
# MATISSE 5NF9 — 3 × 100 ps reviewer analysis

This notebook analyzes the two YASARA outputs:

- `5NF9_MATISSE_all_replicates.txt` — frame-by-frame values
- `5NF9_MATISSE_summary.txt` — macro-generated replicate summary

The main observable is:

\[
V_e = V - V_i
\]

where:

- **V** = summed MATISSE sphere volume
- **Vi** = pairwise intersection volume
- **Ve** = effective MATISSE pocket volume

## Analysis performed

1. Validate and normalize the two input tables.
2. Compare the macro summary with values recalculated from the raw trajectory table.
3. Separate:
   - **0–10 ps**: initial relaxation
   - **10–100 ps**: main analysis
4. For each replicate calculate:
   - mean Ve
   - SD Ve
   - median Ve
   - min/max Ve
   - CV
   - mean Ve in 0–10 ps
   - mean Ve in 10–100 ps
5. Calculate combined mean ± SD across the **three replicate means**.
6. Compare the blocks:
   - 0–10 ps
   - 10–40 ps
   - 40–70 ps
   - 70–100 ps
7. Calculate a **5 ps running average**.
8. Generate:
   - **Figure A:** Ve vs time for all replicates + replicate mean ± SD
   - **Figure B:** post-relaxation Ve distributions (10–100 ps)
   - **Figure C:** 5 ps running averages
   - **Figure D:** block-wise mean Ve
9. Export all tables, plots, and a text report as a ZIP archive.

> **Important:** The short 100 fs YASARA run is only a technical test. If you upload test data that do not reach 10 or 100 ps, the notebook will warn you and skip analyses that require those time windows.


In [ ]:

#@title 1. Analysis settings

RELAX_END_PS = 10.0       #@param {type:"number"}
ANALYSIS_END_PS = 100.0   #@param {type:"number"}
RUNNING_AVG_PS = 5.0      #@param {type:"number"}

# Block definitions used for the stability comparison.
BLOCKS = [
    (0.0, 10.0),
    (10.0, 40.0),
    (40.0, 70.0),
    (70.0, 100.0),
]

# Optional reference value. Leave as 0 if ligand volume has not been calculated.
LIGAND_VOLUME_A3 = 0.0    #@param {type:"number"}

# Plot/export settings
FIG_DPI = 300             #@param {type:"integer"}
OUTPUT_DIR = "MATISSE_analysis_output"

print("Settings loaded.")
print(f"Relaxation: 0–{RELAX_END_PS:g} ps")
print(f"Main analysis: {RELAX_END_PS:g}–{ANALYSIS_END_PS:g} ps")
print(f"Running average: {RUNNING_AVG_PS:g} ps")


In [ ]:

#@title 2. Upload the two MATISSE text files

from google.colab import files
import io
import os
import re
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

uploaded = files.upload()

if len(uploaded) < 2:
    raise RuntimeError(
        "Please upload BOTH files: "
        "5NF9_MATISSE_all_replicates.txt and 5NF9_MATISSE_summary.txt"
    )

def find_uploaded_file(keyword):
    matches = [name for name in uploaded if keyword.lower() in name.lower()]
    if not matches:
        return None
    return matches[0]

raw_name = find_uploaded_file("all_replicates")
summary_name = find_uploaded_file("summary")

if raw_name is None:
    raise RuntimeError("Could not identify the file containing 'all_replicates' in its name.")
if summary_name is None:
    raise RuntimeError("Could not identify the file containing 'summary' in its name.")

print("Raw trajectory file:", raw_name)
print("Summary file:", summary_name)


In [ ]:

#@title 3. Read, normalize, and validate the tables

def read_whitespace_table(file_bytes):
    return pd.read_csv(io.BytesIO(file_bytes), sep=r"\s+", engine="python")

raw_original = read_whitespace_table(uploaded[raw_name])
summary_original = read_whitespace_table(uploaded[summary_name])

# Supports both the older output names and the newer concise macro names.
RAW_ALIASES = {
    "Replicate": "Rep",
    "Rep": "Rep",
    "Seed": "Seed",
    "Snapshot": "Snap",
    "Snap": "Snap",
    "Time_fs": "Time_fs",
    "Time_ps": "Time_ps",
    "LocalWaters": "LocalW",
    "LocalW": "LocalW",
    "MATISSESpheres": "Spheres",
    "Spheres": "Spheres",
    "MotherSphereVolume_A3": "V",
    "V": "V",
    "MotherOverlap_A3": "Vi",
    "Vi": "Vi",
    "EffectiveVolume_A3": "Ve",
    "Ve": "Ve",
}

SUMMARY_ALIASES = {
    "Replicate": "Rep",
    "Rep": "Rep",
    "Seed": "Seed",
    "MeanVe_A3": "Ve",
    "SDVe_A3": "stdVe",
    "MinVe_A3": "minVe",
    "MaxVe_A3": "maxVe",
    "V": "V",
    "stdV": "stdV",
    "Vi": "Vi",
    "stdVi": "stdVi",
    "Ve": "Ve",
    "stdVe": "stdVe",
}

raw = raw_original.rename(columns={c: RAW_ALIASES.get(c, c) for c in raw_original.columns}).copy()
summary = summary_original.rename(columns={c: SUMMARY_ALIASES.get(c, c) for c in summary_original.columns}).copy()

required_raw = ["Rep", "Seed", "Snap", "Time_fs", "Time_ps", "V", "Vi", "Ve"]
missing = [c for c in required_raw if c not in raw.columns]
if missing:
    raise RuntimeError(
        "The raw file is missing required columns after normalization: "
        + ", ".join(missing)
    )

# Convert all relevant fields to numeric.
for col in raw.columns:
    raw[col] = pd.to_numeric(raw[col], errors="ignore")
for col in summary.columns:
    summary[col] = pd.to_numeric(summary[col], errors="ignore")

raw["Rep"] = raw["Rep"].astype(int)
raw["Snap"] = raw["Snap"].astype(int)
if "Rep" in summary.columns:
    summary["Rep"] = summary["Rep"].astype(int)

raw = raw.sort_values(["Rep", "Time_ps", "Snap"]).reset_index(drop=True)

print("\nNormalized raw columns:")
print(list(raw.columns))
print("\nNormalized summary columns:")
print(list(summary.columns))

print("\nReplicates detected:", sorted(raw["Rep"].unique().tolist()))
print(f"Time range: {raw['Time_ps'].min():.4f}–{raw['Time_ps'].max():.4f} ps")
print(f"Total raw frames: {len(raw)}")

if raw["Time_ps"].max() < RELAX_END_PS:
    warnings.warn(
        f"The uploaded trajectory ends at {raw['Time_ps'].max():.4f} ps, "
        f"which is before the requested {RELAX_END_PS:g} ps relaxation boundary. "
        "This looks like a short technical test, not the final 100 ps dataset."
    )
elif raw["Time_ps"].max() < ANALYSIS_END_PS:
    warnings.warn(
        f"The uploaded trajectory ends at {raw['Time_ps'].max():.4f} ps, "
        f"before the requested {ANALYSIS_END_PS:g} ps analysis endpoint."
    )

display(raw.head())
display(summary)


In [ ]:

#@title 4. Validate the macro summary against the raw trajectory table

# Recalculate WHOLE-TRAJECTORY statistics from the raw file.
whole_summary = (
    raw.groupby("Rep", as_index=False)
       .agg(
           V=("V", "mean"),
           stdV=("V", "std"),
           Vi=("Vi", "mean"),
           stdVi=("Vi", "std"),
           Ve=("Ve", "mean"),
           stdVe=("Ve", "std"),
           minVe=("Ve", "min"),
           maxVe=("Ve", "max"),
           nFrames=("Ve", "size"),
       )
)

# Merge only columns that exist in the supplied macro summary.
macro_cols = [c for c in ["Rep", "V", "stdV", "Vi", "stdVi", "Ve", "stdVe", "minVe", "maxVe"] if c in summary.columns]
if "Rep" in macro_cols and len(macro_cols) > 1:
    macro_check = whole_summary.merge(
        summary[macro_cols],
        on="Rep",
        how="left",
        suffixes=("_recalc", "_macro")
    )
else:
    macro_check = whole_summary.copy()

print("Whole-trajectory statistics recalculated from the raw file:")
display(whole_summary)

print("\nMacro summary vs recalculated values:")
display(macro_check)

# Diagnostic: Ve should equal V - Vi frame by frame.
raw["Ve_from_V_minus_Vi"] = raw["V"] - raw["Vi"]
raw["Ve_abs_error"] = (raw["Ve"] - raw["Ve_from_V_minus_Vi"]).abs()

max_identity_error = raw["Ve_abs_error"].max()
print(f"\nMaximum |Ve - (V - Vi)| = {max_identity_error:.6g} Å³")

if max_identity_error > 1e-3:
    warnings.warn("Ve is not numerically equal to V - Vi for some rows. Inspect the macro output.")


In [ ]:

#@title 5. Define relaxation and post-relaxation datasets

# Non-overlapping convention:
# relaxation:      0 <= t < RELAX_END_PS
# main analysis:   RELAX_END_PS <= t <= ANALYSIS_END_PS

relax = raw[
    (raw["Time_ps"] >= 0.0) &
    (raw["Time_ps"] < RELAX_END_PS)
].copy()

analysis = raw[
    (raw["Time_ps"] >= RELAX_END_PS) &
    (raw["Time_ps"] <= ANALYSIS_END_PS)
].copy()

print(f"Relaxation frames (0–{RELAX_END_PS:g} ps): {len(relax)}")
print(f"Main-analysis frames ({RELAX_END_PS:g}–{ANALYSIS_END_PS:g} ps): {len(analysis)}")

if analysis.empty:
    print(
        "\nNOTE: There are no frames in the requested post-relaxation interval. "
        "This is expected for a 100 fs test. Run the 3 × 100 ps production simulation "
        "before interpreting reviewer-level statistics."
    )


In [ ]:

#@title 6. Per-replicate statistics: early vs post-relaxation

def cv_percent(series):
    mean = series.mean()
    if pd.isna(mean) or mean == 0:
        return np.nan
    return series.std(ddof=1) / mean * 100.0

rep_ids = sorted(raw["Rep"].unique())

early_means = (
    relax.groupby("Rep")["Ve"].mean()
    if not relax.empty else pd.Series(dtype=float)
)

if not analysis.empty:
    per_rep = (
        analysis.groupby("Rep")
        .agg(
            nFrames=("Ve", "size"),
            meanVe=("Ve", "mean"),
            stdVe=("Ve", "std"),
            medianVe=("Ve", "median"),
            minVe=("Ve", "min"),
            maxVe=("Ve", "max"),
            meanV=("V", "mean"),
            stdV=("V", "std"),
            meanVi=("Vi", "mean"),
            stdVi=("Vi", "std"),
        )
        .reset_index()
    )
    cvs = analysis.groupby("Rep")["Ve"].apply(cv_percent).rename("CV_percent").reset_index()
    per_rep = per_rep.merge(cvs, on="Rep", how="left")
else:
    per_rep = pd.DataFrame({"Rep": rep_ids})

per_rep["meanVe_0_10ps"] = per_rep["Rep"].map(early_means)

if "meanVe" in per_rep.columns:
    per_rep["meanVe_10_100ps"] = per_rep["meanVe"]
    per_rep["delta_post_minus_early"] = (
        per_rep["meanVe_10_100ps"] - per_rep["meanVe_0_10ps"]
    )
    per_rep["percent_change_early_to_post"] = (
        per_rep["delta_post_minus_early"] /
        per_rep["meanVe_0_10ps"] * 100.0
    )

print("Per-replicate reviewer statistics:")
display(per_rep)


In [ ]:

#@title 7. Combined result across independent replicate means

if not analysis.empty and "meanVe" in per_rep.columns and per_rep["meanVe"].notna().any():
    replicate_means = per_rep["meanVe"].dropna()

    combined = pd.DataFrame({
        "metric": [
            "Mean of replicate mean Ve",
            "SD between replicate mean Ve",
            "Min replicate mean Ve",
            "Max replicate mean Ve",
            "Number of independent replicates",
            "Mean within-replicate SD Ve",
        ],
        "value": [
            replicate_means.mean(),
            replicate_means.std(ddof=1),
            replicate_means.min(),
            replicate_means.max(),
            len(replicate_means),
            per_rep["stdVe"].mean(),
        ]
    })

    combined_mean = replicate_means.mean()
    combined_sd = replicate_means.std(ddof=1)

    print(
        f"Post-relaxation result across independent replicate means: "
        f"{combined_mean:.3f} ± {combined_sd:.3f} Å³ "
        f"(mean ± SD across {len(replicate_means)} replicate means)"
    )

    print(
        "\nInterpretation note: the SD between replicate means describes replicate-to-replicate "
        "reproducibility. Each replicate's own stdVe describes short-timescale fluctuations."
    )

    display(combined)
else:
    combined = pd.DataFrame(columns=["metric", "value"])
    print("Combined post-relaxation statistics cannot yet be calculated from this dataset.")


In [ ]:

#@title 8. Block analysis: 0–10, 10–40, 40–70, 70–100 ps

block_rows = []

for rep in rep_ids:
    rep_df = raw[raw["Rep"] == rep]

    for block_index, (start, end) in enumerate(BLOCKS):
        # Avoid double-counting shared boundaries.
        # Every block except the last is [start, end); last is [start, end].
        is_last = block_index == len(BLOCKS) - 1

        if is_last:
            subset = rep_df[
                (rep_df["Time_ps"] >= start) &
                (rep_df["Time_ps"] <= end)
            ]
        else:
            subset = rep_df[
                (rep_df["Time_ps"] >= start) &
                (rep_df["Time_ps"] < end)
            ]

        if len(subset):
            block_rows.append({
                "Rep": rep,
                "Block": f"{start:g}-{end:g} ps",
                "Start_ps": start,
                "End_ps": end,
                "nFrames": len(subset),
                "meanVe": subset["Ve"].mean(),
                "stdVe": subset["Ve"].std(ddof=1),
                "medianVe": subset["Ve"].median(),
                "minVe": subset["Ve"].min(),
                "maxVe": subset["Ve"].max(),
            })

block_stats = pd.DataFrame(block_rows)

if len(block_stats):
    print("Block-wise Ve statistics:")
    display(block_stats)

    block_across_reps = (
        block_stats.groupby(["Block", "Start_ps", "End_ps"], as_index=False)
        .agg(
            mean_of_rep_means=("meanVe", "mean"),
            SD_between_rep_means=("meanVe", "std"),
            nReplicates=("Rep", "nunique"),
        )
        .sort_values("Start_ps")
    )

    print("\nBlock-wise comparison across replicate means:")
    display(block_across_reps)
else:
    block_across_reps = pd.DataFrame()
    print("No requested time blocks are represented in the uploaded data.")


In [ ]:

#@title 9. Calculate the 5 ps running average

raw_with_runavg = raw.copy()
raw_with_runavg["Ve_runavg"] = np.nan

running_window_info = []

for rep in rep_ids:
    idx = raw_with_runavg.index[raw_with_runavg["Rep"] == rep]
    rep_df = raw_with_runavg.loc[idx].sort_values("Time_ps")

    dt_values = rep_df["Time_ps"].diff().dropna()
    dt_values = dt_values[dt_values > 0]

    if len(dt_values) == 0:
        continue

    dt = float(dt_values.median())
    requested_frames = max(1, int(round(RUNNING_AVG_PS / dt)))

    if rep_df["Time_ps"].max() - rep_df["Time_ps"].min() < RUNNING_AVG_PS:
        running_window_info.append(
            (rep, dt, requested_frames, False)
        )
        continue

    runavg = (
        rep_df["Ve"]
        .rolling(
            window=requested_frames,
            center=True,
            min_periods=max(2, requested_frames // 2)
        )
        .mean()
    )

    raw_with_runavg.loc[rep_df.index, "Ve_runavg"] = runavg.values
    running_window_info.append(
        (rep, dt, requested_frames, True)
    )

print("Running-average setup:")
for rep, dt, frames, calculated in running_window_info:
    print(
        f"Rep {rep}: Δt ≈ {dt:.4g} ps; "
        f"{RUNNING_AVG_PS:g} ps ≈ {frames} frames; "
        f"{'calculated' if calculated else 'skipped (trajectory too short)'}"
    )


In [ ]:

#@title 10. Create output directory

outdir = Path(OUTPUT_DIR)
outdir.mkdir(parents=True, exist_ok=True)

print("Output directory:", outdir.resolve())


In [ ]:

#@title 11. Figure A — Ve vs time, three replicates + mean ± SD

fig, ax = plt.subplots(figsize=(10, 6))

for rep, grp in raw.groupby("Rep"):
    ax.plot(
        grp["Time_ps"],
        grp["Ve"],
        linewidth=1.1,
        alpha=0.55,
        label=f"Replicate {rep}"
    )

# Mean and SD across replicates at each common time point.
time_stats = (
    raw.groupby("Time_ps", as_index=False)["Ve"]
       .agg(["mean", "std"])
       .reset_index()
)

ax.plot(
    time_stats["Time_ps"],
    time_stats["mean"],
    linewidth=2.5,
    label="Replicate mean"
)

if time_stats["std"].notna().any():
    ax.fill_between(
        time_stats["Time_ps"],
        time_stats["mean"] - time_stats["std"].fillna(0),
        time_stats["mean"] + time_stats["std"].fillna(0),
        alpha=0.18,
        label="± SD across replicates"
    )

if raw["Time_ps"].max() >= RELAX_END_PS:
    ax.axvline(
        RELAX_END_PS,
        linestyle="--",
        linewidth=1.5,
        label=f"Relaxation boundary ({RELAX_END_PS:g} ps)"
    )

ax.set_xlabel("Time (ps)")
ax.set_ylabel(r"$V_e$ ($\AA^3$)")
ax.set_title("MATISSE-derived pocket volume over time")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()

figA = outdir / "Figure_A_Ve_time_series.png"
fig.savefig(figA, dpi=FIG_DPI, bbox_inches="tight")
plt.show()

print("Saved:", figA)


In [ ]:

#@title 12. Figure B — post-relaxation Ve distributions

if analysis.empty:
    print(
        f"Figure B skipped: no data are available in the "
        f"{RELAX_END_PS:g}–{ANALYSIS_END_PS:g} ps interval."
    )
else:
    reps = sorted(analysis["Rep"].unique())
    datasets = [
        analysis.loc[analysis["Rep"] == rep, "Ve"].dropna().to_numpy()
        for rep in reps
    ]

    fig, ax = plt.subplots(figsize=(8, 6))

    vp = ax.violinplot(
        datasets,
        positions=np.arange(1, len(reps) + 1),
        showmeans=False,
        showmedians=True,
        showextrema=True
    )

    # Overlay a conventional box plot for quartiles.
    ax.boxplot(
        datasets,
        positions=np.arange(1, len(reps) + 1),
        widths=0.18,
        showfliers=False
    )

    ax.set_xticks(np.arange(1, len(reps) + 1))
    ax.set_xticklabels([f"Replicate {rep}" for rep in reps])
    ax.set_ylabel(r"$V_e$ ($\AA^3$)")
    ax.set_title(
        f"Post-relaxation MATISSE pocket-volume distributions "
        f"({RELAX_END_PS:g}–{ANALYSIS_END_PS:g} ps)"
    )
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()

    figB = outdir / "Figure_B_Ve_distribution_10_100ps.png"
    fig.savefig(figB, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

    print("Saved:", figB)


In [ ]:

#@title 13. Figure C — 5 ps running averages

runavg_available = raw_with_runavg["Ve_runavg"].notna().any()

if not runavg_available:
    print(
        f"Figure C skipped: the trajectory is too short for a "
        f"{RUNNING_AVG_PS:g} ps running average."
    )
else:
    fig, ax = plt.subplots(figsize=(10, 6))

    for rep, grp in raw_with_runavg.groupby("Rep"):
        ax.plot(
            grp["Time_ps"],
            grp["Ve_runavg"],
            linewidth=2,
            label=f"Replicate {rep}"
        )

    if raw["Time_ps"].max() >= RELAX_END_PS:
        ax.axvline(
            RELAX_END_PS,
            linestyle="--",
            linewidth=1.5,
            label=f"Relaxation boundary ({RELAX_END_PS:g} ps)"
        )

    ax.set_xlabel("Time (ps)")
    ax.set_ylabel(r"Running mean $V_e$ ($\AA^3$)")
    ax.set_title(f"{RUNNING_AVG_PS:g} ps running average of MATISSE pocket volume")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()

    figC = outdir / "Figure_C_Ve_running_average.png"
    fig.savefig(figC, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

    print("Saved:", figC)


In [ ]:

#@title 14. Figure D — block-wise mean Ve

if block_stats.empty:
    print("Figure D skipped: no requested analysis blocks are represented.")
else:
    fig, ax = plt.subplots(figsize=(9, 6))

    block_order = (
        block_stats[["Block", "Start_ps"]]
        .drop_duplicates()
        .sort_values("Start_ps")
    )
    ordered_labels = block_order["Block"].tolist()
    x = np.arange(len(ordered_labels))

    for rep in sorted(block_stats["Rep"].unique()):
        temp = (
            block_stats[block_stats["Rep"] == rep]
            .set_index("Block")
            .reindex(ordered_labels)
        )
        ax.plot(
            x,
            temp["meanVe"],
            marker="o",
            linewidth=1.8,
            label=f"Replicate {rep}"
        )

    ax.set_xticks(x)
    ax.set_xticklabels(ordered_labels)
    ax.set_xlabel("Trajectory interval")
    ax.set_ylabel(r"Mean $V_e$ ($\AA^3$)")
    ax.set_title("Block-wise stability of the MATISSE-derived pocket volume")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()

    figD = outdir / "Figure_D_Ve_block_means.png"
    fig.savefig(figD, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

    print("Saved:", figD)


In [ ]:

#@title 15. Optional ligand-volume reference

if LIGAND_VOLUME_A3 > 0:
    print(f"Ligand molecular volume supplied: {LIGAND_VOLUME_A3:.3f} Å³")

    if not analysis.empty and "meanVe" in per_rep.columns:
        post_mean = per_rep["meanVe"].mean()
        between_rep_sd = per_rep["meanVe"].std(ddof=1)

        print(
            f"Mean post-relaxation MATISSE Ve across replicate means: "
            f"{post_mean:.3f} ± {between_rep_sd:.3f} Å³"
        )
        print(
            "The ligand volume is a physical reference scale only; "
            "it is not expected to equal the water-accessible pocket volume."
        )
else:
    print(
        "No ligand molecular volume was supplied. "
        "It cannot be calculated from the two MATISSE text files alone."
    )


In [ ]:

#@title 16. Export numerical tables

raw.to_csv(outdir / "raw_normalized.csv", index=False)
raw_with_runavg.to_csv(outdir / "raw_with_running_average.csv", index=False)
whole_summary.to_csv(outdir / "whole_trajectory_recalculated_summary.csv", index=False)
macro_check.to_csv(outdir / "macro_vs_recalculated_summary.csv", index=False)
per_rep.to_csv(outdir / "post_relaxation_per_replicate_statistics.csv", index=False)
combined.to_csv(outdir / "combined_replicate_statistics.csv", index=False)
block_stats.to_csv(outdir / "block_statistics_per_replicate.csv", index=False)
block_across_reps.to_csv(outdir / "block_statistics_across_replicates.csv", index=False)

print("Exported numerical tables:")
for p in sorted(outdir.glob("*.csv")):
    print(" -", p.name)


In [ ]:

#@title 17. Generate an automatic analysis report

report_lines = []

report_lines.append("MATISSE 5NF9 trajectory analysis")
report_lines.append("=" * 60)
report_lines.append(f"Input raw file: {raw_name}")
report_lines.append(f"Input summary file: {summary_name}")
report_lines.append(f"Replicates detected: {', '.join(map(str, rep_ids))}")
report_lines.append(
    f"Observed trajectory range: {raw['Time_ps'].min():.4f}–{raw['Time_ps'].max():.4f} ps"
)
report_lines.append(
    f"Requested relaxation interval: 0–{RELAX_END_PS:g} ps"
)
report_lines.append(
    f"Requested main analysis interval: {RELAX_END_PS:g}–{ANALYSIS_END_PS:g} ps"
)
report_lines.append("")
report_lines.append(
    f"Maximum |Ve - (V - Vi)|: {max_identity_error:.6g} Å^3"
)

if analysis.empty:
    report_lines.append("")
    report_lines.append(
        "The uploaded dataset does not reach the post-relaxation analysis interval. "
        "No reviewer-level 10–100 ps interpretation was generated."
    )
else:
    report_lines.append("")
    report_lines.append("POST-RELAXATION PER-REPLICATE RESULTS")
    report_lines.append("-" * 60)

    for _, row in per_rep.iterrows():
        report_lines.append(
            f"Replicate {int(row['Rep'])}: "
            f"Ve = {row['meanVe']:.3f} ± {row['stdVe']:.3f} Å^3; "
            f"median = {row['medianVe']:.3f}; "
            f"range = {row['minVe']:.3f}–{row['maxVe']:.3f}; "
            f"CV = {row['CV_percent']:.2f}%."
        )
        if pd.notna(row.get("meanVe_0_10ps", np.nan)):
            report_lines.append(
                f"  Mean Ve 0–{RELAX_END_PS:g} ps = {row['meanVe_0_10ps']:.3f} Å^3; "
                f"mean Ve {RELAX_END_PS:g}–{ANALYSIS_END_PS:g} ps = "
                f"{row['meanVe_10_100ps']:.3f} Å^3."
            )

    replicate_means = per_rep["meanVe"].dropna()
    if len(replicate_means):
        report_lines.append("")
        report_lines.append(
            f"Across independent replicate means: "
            f"Ve = {replicate_means.mean():.3f} ± "
            f"{replicate_means.std(ddof=1):.3f} Å^3 "
            f"(mean ± SD between replicate means)."
        )

    if len(block_stats):
        report_lines.append("")
        report_lines.append("BLOCK MEANS")
        report_lines.append("-" * 60)
        for rep in rep_ids:
            temp = block_stats[block_stats["Rep"] == rep].sort_values("Start_ps")
            if len(temp):
                formatted = "; ".join(
                    f"{r.Block}: {r.meanVe:.3f} Å^3"
                    for r in temp.itertuples()
                )
                report_lines.append(f"Replicate {rep}: {formatted}")

if LIGAND_VOLUME_A3 > 0:
    report_lines.append("")
    report_lines.append(f"Ligand molecular volume reference: {LIGAND_VOLUME_A3:.3f} Å^3")

report_path = outdir / "MATISSE_analysis_report.txt"
report_path.write_text("\n".join(report_lines), encoding="utf-8")

print(report_path.read_text())


In [ ]:

#@title 18. Create ZIP archive and download all outputs

archive_base = "MATISSE_analysis_output"
archive_path = shutil.make_archive(
    archive_base,
    "zip",
    root_dir=outdir
)

print("Created:", archive_path)

files.download(archive_path)



## How to interpret the production 100 ps result

The desirable pattern is **not a perfectly constant pocket volume**. Instead, look for:

\[
\text{initial directional relaxation}
\rightarrow
\text{fluctuations around a comparatively stable mean}
\]

Evidence supporting this interpretation would include:

- a systematic difference between 0–10 ps and 10–100 ps;
- comparable 10–40, 40–70, and 70–100 ps block means;
- a 5 ps running average that becomes relatively stationary after the initial period;
- substantial overlap among the three post-relaxation Ve distributions;
- similar mean Ve values among the three independent trajectories.

If the block means continue to move systematically or the three replicates occupy very different volume ranges, do **not** claim stabilization. In that case, the correct conclusion is that 100 ps did not establish a stationary short-timescale MATISSE volume regime.
